# ALS Collaborative Filtering - Batch Scoring
### Loads Production model from bharatmart.ml.bharatmart_als_model
### Generates Top-10 recommendations per customer
### Writes to bharatmart.ml.gld_als_recommendations

In [0]:
from pyspark.sql.functions import col, lit, explode, current_date
from pyspark.ml.recommendation import ALSModel
from pyspark.ml import PipelineModel
import mlflow
import mlflow.spark

mlflow.autolog(disable=True)

print("imports done")

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS bharatmart.ml.tmp_volume")
print("volume created")

**Load Production model and indexer**

In [0]:
import os
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/bharatmart/ml/tmp_volume"

model_uri  = "models:/bharatmart.ml.bharatmart_als_model@Production"
best_model = mlflow.spark.load_model(model_uri)

print("model loaded")
print(f"type : {type(best_model)}")

In [0]:
from pyspark.ml.recommendation import ALSModel

# extract ALSModel from inside the PipelineModel wrapper
als_model = best_model.stages[-1]

print(f"type  : {type(als_model)}")
print(f"rank  : {als_model.rank}")

**Load silver tables**

In [0]:
orders_df  = spark.table("bharatmart.silver.slv_orders")
cart_df    = spark.table("bharatmart.silver.slv_cart_events")
reviews_df = spark.table("bharatmart.silver.slv_reviews")

print(f"orders : {orders_df.count():,}")
print(f"cart : {cart_df.count():,}")
print(f"reviews : {reviews_df.count():,}")

**Build interaction matrix**

In [0]:
from pyspark.sql.functions import lit
from pyspark.sql.functions import max as spark_max

# clean orders
clean_orders = orders_df.filter(
    col("customer_id").isNotNull() & col("product_id").isNotNull()
)

# clean reviews
clean_reviews = reviews_df.filter(
    (col("_is_ghost_order") == False) & (col("rating") >= 4)
)

# three signals
purchase_df = clean_orders.select(
    col("customer_id"), col("product_id"), lit(1.0).alias("confidence")
)

cart_signal_df = cart_df.filter(
    col("action").isin("cart_add", "wishlist_add")
).select(
    col("customer_id"), col("product_id"), lit(0.5).alias("confidence")
)

review_signal_df = clean_reviews.select(
    col("customer_id"), col("product_id"), lit(2.0).alias("confidence")
)

# merge
interaction_df = purchase_df.union(cart_signal_df).union(review_signal_df) \
    .groupBy("customer_id", "product_id") \
    .agg(spark_max("confidence").alias("confidence"))

print(f"interaction pairs : {interaction_df.count():,}")

**fit StringIndexer on current data**

In [0]:
from pyspark.sql.functions import count, dense_rank
from pyspark.sql.window import Window

# StringIndexer orders by frequency descending — replicate that exactly
user_freq = interaction_df.groupBy("customer_id") \
    .agg(count("*").alias("freq")) \
    .orderBy(col("freq").desc(), col("customer_id"))

item_freq = interaction_df.groupBy("product_id") \
    .agg(count("*").alias("freq")) \
    .orderBy(col("freq").desc(), col("product_id"))

user_window = Window.orderBy(col("freq").desc(), col("customer_id"))
item_window = Window.orderBy(col("freq").desc(), col("product_id"))

indexed_df = interaction_df \
    .join(user_freq, "customer_id") \
    .withColumn("user_idx", (dense_rank().over(user_window) - 1).cast("integer")) \
    .drop("freq") \
    .join(item_freq, "product_id") \
    .withColumn("item_idx", (dense_rank().over(item_window) - 1).cast("integer")) \
    .drop("freq")

print(f"indexed rows : {indexed_df.count():,}")

**Generate Top-10 recommendations per user**

In [0]:
import numpy as np
import pandas as pd

# extract user and item factor matrices from trained model
user_factors_pdf = als_model.userFactors.toPandas()
item_factors_pdf = als_model.itemFactors.toPandas()

user_ids = user_factors_pdf["id"].values
item_ids = item_factors_pdf["id"].values

U = np.array(user_factors_pdf["features"].tolist())  # users x rank
V = np.array(item_factors_pdf["features"].tolist())  # items x rank

print(f"user factors shape : {U.shape}")
print(f"item factors shape : {V.shape}")

**Explode recommendations into one row per user-item pair**

In [0]:
batch_size = 1000
all_recs   = []

for i in range(0, len(user_ids), batch_size):
    batch_U    = U[i:i+batch_size]
    scores     = batch_U @ V.T  # (batch, n_items)

    for j, user_idx in enumerate(user_ids[i:i+batch_size]):
        user_scores  = scores[j]
        top10_idx    = np.argsort(user_scores)[::-1][:10]  # proper sort descending
        for rank, k in enumerate(top10_idx, 1):
            all_recs.append((
                int(user_idx),
                int(item_ids[k]),
                rank,
                float(user_scores[k])
            ))

recs_pdf = pd.DataFrame(all_recs, columns=["user_idx","item_idx","rank","als_score"])
print(f"recommendations generated : {len(recs_pdf):,}")

**Map integer indices back to string IDs**

In [0]:
# build lookup dicts directly from indexed_df
user_lookup = indexed_df.select("user_idx", "customer_id").distinct().toPandas()
item_lookup  = indexed_df.select("item_idx", "product_id").distinct().toPandas()

user_idx_to_id = dict(zip(user_lookup["user_idx"], user_lookup["customer_id"]))
item_idx_to_id = dict(zip(item_lookup["item_idx"], item_lookup["product_id"]))

recs_pdf["customer_id"] = recs_pdf["user_idx"].map(user_idx_to_id)
recs_pdf["product_id"]  = recs_pdf["item_idx"].map(item_idx_to_id)

recs_pdf = recs_pdf.dropna(subset=["customer_id", "product_id"])

print(f"rows after mapping : {len(recs_pdf):,}")
print(recs_pdf[["customer_id","product_id","rank","als_score"]].head(5))

**Add cold start customers with popularity fallback**

In [0]:
from pyspark.sql.functions import count

# customers who had fewer than 3 orders get popularity fallback
all_customers = spark.table("bharatmart.silver.slv_orders") \
    .filter(col("customer_id").isNotNull()) \
    .groupBy("customer_id") \
    .agg(count("order_id").alias("order_count"))

cold_start_customers = all_customers.filter(col("order_count") < 3) \
    .select("customer_id").toPandas()

# top 10 most purchased products as fallback
top10_products = clean_orders.groupBy("product_id") \
    .agg(count("order_id").alias("cnt")) \
    .orderBy(col("cnt").desc()) \
    .limit(10) \
    .select("product_id") \
    .toPandas()["product_id"].tolist()

# build fallback rows
fallback_rows = []
for cust in cold_start_customers["customer_id"]:
    for rank, prod in enumerate(top10_products, 1):
        fallback_rows.append((cust, prod, rank, 0.0, "popularity_fallback"))

fallback_pdf = pd.DataFrame(
    fallback_rows,
    columns=["customer_id","product_id","rank","als_score","rec_source"]
)

print(f"cold start customers : {len(cold_start_customers):,}")
print(f"fallback rows : {len(fallback_pdf):,}")

**Add rec_source to ALS rows and combine with fallback**

In [0]:
recs_pdf["rec_source"] = "als"

final_recs_pdf = pd.concat(
    [recs_pdf[["customer_id","product_id","rank","als_score","rec_source"]],
     fallback_pdf],
    ignore_index=True
)

print(f"als rows : {len(recs_pdf):,}")
print(f"fallback rows : {len(fallback_pdf):,}")
print(f"total rows : {len(final_recs_pdf):,}")

**Convert to Spark and add scored_date**

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import current_date

schema = StructType([
    StructField("customer_id", StringType(),  False),
    StructField("product_id",  StringType(),  False),
    StructField("rank",   IntegerType(), False),
    StructField("als_score",   DoubleType(),  False),
    StructField("rec_source",  StringType(),  False)
])

output_df = spark.createDataFrame(final_recs_pdf, schema=schema) \
    .withColumn("scored_date", current_date())

print(f"output rows : {output_df.count():,}")
display(output_df.limit(5))

**Write to gld_als_recommendations**

In [0]:
output_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bharatmart.ml.gld_als_recommendations")

count = spark.table("bharatmart.ml.gld_als_recommendations").count()
print(f"wrote {count:,} rows to bharatmart.ml.gld_als_recommendations")

**Verify output**

In [0]:
display(spark.table("bharatmart.ml.gld_als_recommendations").limit(10))

**Precision@10 and NDCG@10**

In [0]:
from pyspark.sql.functions import count

# rebuild test set from 11b split
from pyspark.sql.functions import percentile_approx, months_between, current_date

order_dates = clean_orders.select("customer_id", "product_id", "order_date")
indexed_with_date = indexed_df.join(order_dates, ["customer_id", "product_id"], "left")

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
w = Window.partitionBy("customer_id", "product_id").orderBy(col("order_date").desc())
indexed_deduped = indexed_with_date.withColumn("rn", row_number().over(w)) \
    .filter(col("rn") == 1).drop("rn")

cutoff_date = indexed_deduped.select(
    percentile_approx("order_date", 0.7).alias("cutoff_date")
).toPandas()["cutoff_date"][0]

test_df = indexed_deduped.filter(col("order_date") > cutoff_date)

# als recommendations (no fallback)
als_recs_df = spark.table("bharatmart.ml.gld_als_recommendations") \
    .filter(col("rec_source") == "als")

# actual purchases in test set
actual = test_df.select("customer_id", "product_id").distinct()

# hits
hits_df = als_recs_df.join(actual, ["customer_id", "product_id"], "inner")
hits      = hits_df.count()
n_users   = als_recs_df.select("customer_id").distinct().count()

precision_at_10 = hits / n_users
print(f"hits : {hits:,}")
print(f"users scored  : {n_users:,}")
print(f"precision@10  : {precision_at_10:.4f}")
print(f"random baseline : {10/47030:.4f}")

**NDCG@10**

In [0]:
import numpy as np

# join rank info to hits
hits_with_rank = als_recs_df.join(actual, ["customer_id", "product_id"], "inner") \
    .select("customer_id", "rank")

hits_pdf = hits_with_rank.toPandas()

# NDCG per user
def dcg(ranks):
    return sum(1 / np.log2(r + 1) for r in ranks)

ideal_dcg = dcg(range(1, 11))  # perfect: all 10 ranks are hits

user_dcg = hits_pdf.groupby("customer_id")["rank"].apply(list).apply(dcg)
ndcg_per_user = user_dcg / ideal_dcg

# average over all users including those with 0 hits
ndcg_at_10 = ndcg_per_user.sum() / n_users
print(f"ndcg@10 : {ndcg_at_10:.4f}")

**Inversion check and segment sample**

In [0]:
from pyspark.sql.functions import col, lit, explode, current_date, lag

# inversion check
inversion_check = spark.table("bharatmart.ml.gld_als_recommendations") \
    .filter(col("rec_source") == "als")

w2 = Window.partitionBy("customer_id").orderBy("rank")
inv_df = inversion_check \
    .withColumn("prev_score", lag("als_score").over(w2)) \
    .filter(col("prev_score").isNotNull()) \
    .filter(col("als_score") > col("prev_score"))

print(f"inversion count : {inv_df.count():,}")

# sample recs for one customer per RFM segment
rfm  = spark.table("bharatmart.ml.gld_rfm_segments").select("customer_id", "rfm_segment")
recs = spark.table("bharatmart.ml.gld_als_recommendations").filter(col("rank") <= 3)

sample_customers = rfm.groupBy("rfm_segment") \
    .agg(spark_max("customer_id").alias("customer_id"))

sample_recs = recs \
    .join(sample_customers, "customer_id") \
    .join(rfm.withColumnRenamed("rfm_segment", "segment"), "customer_id") \
    .select("segment", "customer_id", "product_id", "rank", "als_score") \
    .orderBy("segment", "rank")

display(sample_recs)